In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random
from faker import Faker

# paths tools
import os
import sys

# making src visible from here
src_root = os.path.abspath("..")
if src_root not in  sys.path:
    sys.path.append(src_root)

# custom
import src.utils as utils
import src.models as models

# imports re for text cleaning
import re
from datetime import datetime, timedelta, date

# we will ignore pandas warning
import warnings
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# paths tools
import os
import sys

# making src visible from here
os.path.abspath("..")

'/Users/andreypetukhov/Documents/work-related/LightFM'

## Generating dataframes

In [10]:
# all lightfm imports 
from lightfm.data import Dataset
from lightfm import LightFM
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k
from lightfm.evaluation import auc_score

In [11]:
# i recommend setting num_transactions=50000 in case of cloud computations
# for larger sample size
# adapt the num_inns and num_groups to your needs
df, inns, inn2group, group2name = utils.make_interactions_dataset(
    num_transactions=1000,
    num_inns=80,
    num_groups=10
)

df_train, df_test = train_test_split(df, test_size=0.2)

# dropping duplicate interactions from test which are present in train
train_kt_dt_set = set(zip(df_train['inn_kt'], df_train['inn_dt']))
df_test = df_test[~df_test[['inn_kt', 'inn_dt']].apply(tuple, axis=1).isin(train_kt_dt_set)]
df_test = df_test.reset_index(drop=True)

df.head()

,id_trans,inn_kt,inn_dt,c_sum,date,nazn,kt_group_num,dt_group_num,kt_group_name,dt_group_name,raw_word,word
0,1,6352582168,3882220151,144241.78,2023-03-18,сыр,4,4,from,from,word1,None
1,2,7886942023,2862868358,243469.67,2023-12-17,сыр,10,9,top,simply,None,None
2,3,8004169970,3129711236,209596.56,2021-04-23,колесо,1,1,day,day,None,None
3,4,7855581357,8871760899,36236.36,2022-12-23,хлеб,9,9,simply,simply,word4,None
4,5,2121156710,5695027889,230952.06,2021-10-03,масло,7,7,share,share,word2,clean_2


In [12]:
kt_feature_list = utils.generate_feature_list(df_train, ["kt_group_num", "kt_group_name"])
dt_feature_list = utils.generate_feature_list(df_test, ["dt_group_num", "dt_group_name"])

In [27]:
# features = df.head(10)[["kt_group_num", "kt_group_name"]].apply(lambda x: ",".join(x.map(str)), axis=1)
# features = features.str.split(",")
# features = features.apply(pd.Series).stack().reset_index(drop=True)
# features

In [38]:
# TODO: refactor create_features to pyspark
# TODO: refactor to a class and add order attribute which
# would hold the features order, e.g. {0: "kt_group_num", 1: "kt_group_name"}
utils.create_features(df_train.head(10), ["kt_group_num", "kt_group_name"], "inn_kt")

[(5541583659, ['3', 'doctor']),
 (2410936684, ['9', 'simply']),
 (3579475812, ['4', 'from']),
 (7257059813, ['6', 'read']),
 (1269915191, ['4', 'from']),
 (3579475812, ['4', 'from']),
 (6028495059, ['7', 'share']),
 (9198020798, ['1', 'day']),
 (7417440724, ['8', 'will']),
 (2862868358, ['9', 'simply'])]

In [39]:
user_feature_mapping

{8871760899: 0,
 9596440068: 1,
 5016774679: 2,
 6352582168: 3,
 6325405734: 4,
 3374454823: 5,
 4781048366: 6,
 1211748911: 7,
 4022900276: 8,
 6262290997: 9,
 1269915191: 10,
 5648071742: 11,
 1215563362: 12,
 4653535332: 13,
 2316811877: 14,
 2121156710: 15,
 9319392872: 16,
 2340040298: 17,
 2453113968: 18,
 3882220151: 19,
 6116960896: 20,
 3129711236: 21,
 8951193222: 22,
 1424864397: 23,
 8072361116: 24,
 7855581357: 25,
 5695027889: 26,
 5434714293: 27,
 5411793077: 28,
 9198020798: 29,
 8870658752: 30,
 4661914308: 31,
 5010160842: 32,
 6028495059: 33,
 5317279456: 34,
 3654671079: 35,
 6424988393: 36,
 8004169970: 37,
 6546712311: 38,
 8335373049: 39,
 2511747336: 40,
 5778173193: 41,
 1436211465: 42,
 8232620301: 43,
 9174131477: 44,
 5548535583: 45,
 4065540385: 46,
 9822915880: 47,
 5541583659: 48,
 7994916140: 49,
 4073071417: 50,
 4670673217: 51,
 7886942023: 52,
 5948747606: 53,
 3579475812: 54,
 9955869546: 55,
 2410936684: 56,
 4909552494: 57,
 2862868358: 58,
 557462

In [31]:
# creating dataset
dataset = Dataset()
dataset.fit(
    users=set(inns),
    items=set(inns),
    user_features=kt_feature_list,
    item_features=dt_feature_list
)

interactions, weights = dataset.build_interactions(
    data=list(zip(df_train["inn_kt"], df_train["inn_dt"]))
)

test_interactions, test_weights = dataset.build_interactions(
    data=list(zip(df_test["inn_kt"], df_test["inn_dt"]))
)

# now we are building our questions and professionals features
# in a way that lightfm understand.
# we are using lightfm build in method for building
# questions and professionals features

# TODO: refactor create_features to pyspark
kt_features = dataset.build_user_features(
    utils.create_features(df_train, ["kt_group_num", "kt_group_name"], "inn_kt"),
    normalize=True,
)

dt_features = dataset.build_item_features(
    utils.create_features(df_train, ["dt_group_num", "dt_group_name"], "inn_dt"),
    normalize=True,
)

Колонки из `dt_features` соответствуют `dataset.mapping()` (четверка словарей для айтемов и юзеров).

**NOTE**: `dataset.mapping()` возвращает четверку:

`_user_id_mapping`,

`_user_feature_mapping`,

`_item_id_mapping`,

`_item_feature_mapping`

**Например**, `_item_feature_mapping["Pharmacologist"]` возвращает индекс фичи `"Pharmacologist"` из `dt_features` матрицы, поэтому `dt_features[:, index]` показывает значение фичи "Pharmacologist" для всех айтемов.

# Fitting the model

In [32]:
model = LightFM(
    no_components=150,
    learning_rate=0.05,
    loss='warp',
    random_state=2024)

model.fit(
    interactions,
    item_features=dt_features,
    user_features=kt_features, sample_weight=weights,
    epochs=5, num_threads=1, verbose=True)

print(utils.calculate_auc_score(model, test_interactions, kt_features, dt_features))
print(utils.calculate_precision_at_k(model, test_interactions, kt_features, dt_features))

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
0.5245545
0.020000001


# Making embedding mappings for dt and kt

In [33]:
kt_biases, kt_embeds = model.get_user_representations(kt_features)
dt_biases, dt_embeds = model.get_item_representations(dt_features)
user_id_mapping, user_feature_mapping, item_id_mapping, item_feature_mapping = dataset.mapping()

embeddings = {}
for inn in inns:
    # e.g.
    # user_id_mapping[inn] - index of user in kt_embeds
    embeddings[inn] = {
        "kt_embed": kt_embeds[user_id_mapping[inn]],
        "dt_embed": dt_embeds[item_id_mapping[inn]],
        "kt_bias": kt_biases[user_id_mapping[inn]],
        "dt_bias": dt_biases[item_id_mapping[inn]]
    }

# Obtaining predictions manually

In [54]:
from typing import Dict
# раз ВСЕХ, то и себя тоже что ли?
kt_scores: Dict[int, Dict[int, float]] = {}
# для каждого инн_кт получаю его эмбеддинг и биас, 
# затем для каждого возможного контрагента получаю его эмбед и биас
# считаю скор и складываю в словарь вида инн_кт: {инн_дт1: скор1, инн_дт2: скор2}
for inn_kt in df_test["inn_kt"]:
    inn_idx = user_id_mapping[inn_kt]
    kt_embed = embeddings[inn_kt]["kt_embed"]
    kt_bias = embeddings[inn_kt]["kt_bias"]
    scores = {}
    for inn_dt in inns:
        dt_idx = item_id_mapping[inn_dt]
        dt_embed = embeddings[inn_dt]["dt_embed"]
        dt_bias = embeddings[inn_dt]["dt_bias"]

        score = (dt_embed @ kt_embed) + kt_bias + dt_bias
        scores[inn_dt] = score

    kt_scores[inn_kt] = scores

Получаем все взаимодействия (и на трейне и на тесте (?))

In [99]:
test_kt_dt_set = set(zip(df_test['inn_kt'], df_test['inn_dt']))
train_kt_dt_set = set(zip(df_train['inn_kt'], df_train['inn_dt']))
all_set = train_kt_dt_set | test_kt_dt_set

Считаем map

Попробуем рекомендовать строго тех с кем ещё не взаимодействовали, поскольку на практике их заведемо не будет в тестовом наборе - рекомендовать инн-у тех с кем он взаимодействовал не имеет смысла.

In [112]:
# kt датафрейм с предиктами
kt_preds = pd.DataFrame(kt_scores)

# считаем мапы
average_precisions = {}
average_precisions_no_index = []
top_k = 20
for inn_kt in kt_scores:
    # составляем тех с кем ещё не взаимодействовали в трейне
    data = kt_preds[inn_kt].sort_values(ascending=False)
    data = data[[(inn_kt, inn_dt) not in train_kt_dt_set for inn_dt in data.index]][:top_k]
    top_k_dt = data.index
    top_k_scores = data.values

    # считаем мапы
    targets = np.array([(inn_kt, inn_dt) in test_kt_dt_set for inn_dt in top_k_dt])
    total_ones = sum(targets)
    
    precisions = (targets.cumsum() / np.arange(1, top_k + 1)) * targets
    if total_ones == 0:
        average_precision = 0
    else:
        average_precision = precisions.sum() / total_ones
    
    average_precisions[inn_kt] = average_precision
    average_precisions_no_index.append(average_precision)

average_precisions_no_index = np.array(average_precisions_no_index)


print(average_precisions_no_index.mean())

0.1416732881657141


# Testing PySpark

In [6]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

In [5]:
df

,id_trans,inn_kt,inn_dt,c_sum,date,nazn,kt_group_num,dt_group_num,kt_group_name,dt_group_name,raw_word,word
0,1,8981234834,3301971187,77435.26,2023-11-25,сок,10,10,five,five,None,None
1,2,7470186505,1623327466,102889.58,2021-07-08,сок,9,9,could,could,word2,clean_2
2,3,6590732763,8915621396,24276.97,2022-06-05,молоко,2,2,thought,thought,None,None
3,4,7942681230,4072190906,200773.36,2022-12-05,шоколад,5,5,dark,dark,None,None
4,5,5953657113,9203184457,172017.63,2021-08-18,овощи,7,7,future,future,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,5983848965,5290377866,64220.28,2023-10-21,кофе,6,6,according,according,None,None
996,997,4967848901,4070356474,19977.51,2023-08-04,мясо,7,7,future,future,word1,clean_1
997,998,3501493672,2920078456,48059.55,2022-08-20,овощи,8,8,tree,tree,word3,None
998,999,5953657113,2600595591,48307.45,2021-10-21,фрукты,7,7,future,future,word4,clean_4


In [7]:
spark = SparkSession.builder.appName("TestOperations").getOrCreate()
spark_df = spark.createDataFrame(df)
spark_df.show()

24/12/15 23:16:41 WARN Utils: Your hostname, MacBook-Pro-Andrej.local resolves to a loopback address: 127.0.0.1; using 192.168.1.81 instead (on interface en0)
24/12/15 23:16:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/15 23:17:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------+----------+----------+---------+----------+--------+------------+------------+-------------+-------------+--------+-------+
|id_trans|    inn_kt|    inn_dt|    c_sum|      date|    nazn|kt_group_num|dt_group_num|kt_group_name|dt_group_name|raw_word|   word|
+--------+----------+----------+---------+----------+--------+------------+------------+-------------+-------------+--------+-------+
|       1|8981234834|3301971187| 77435.26|2023-11-25|     сок|          10|          10|         five|         five|    NULL|   NULL|
|       2|7470186505|1623327466|102889.58|2021-07-08|     сок|           9|           9|        could|        could|   word2|clean_2|
|       3|6590732763|8915621396| 24276.97|2022-06-05|  молоко|           2|           2|      thought|      thought|    NULL|   NULL|
|       4|7942681230|4072190906|200773.36|2022-12-05| шоколад|           5|           5|         dark|         dark|    NULL|   NULL|
|       5|5953657113|9203184457|172017.63|2021-08-18|   овощи|

In [8]:
result = spark_df.groupBy().agg(
    count("*").alias("total_deals"),
    count(when(col("raw_word").isNotNull(), 1)).alias("raw_word_count"),
    count(when(col("word").isNotNull(), 1)).alias("word_count")
).withColumn(
    "conversion_raw_word", col("raw_word_count") / col("total_deals")
).withColumn(
    "conversion_word", col("word_count") / col("raw_word_count")
)

result.show()

+-----------+--------------+----------+-------------------+------------------+
|total_deals|raw_word_count|word_count|conversion_raw_word|   conversion_word|
+-----------+--------------+----------+-------------------+------------------+
|       1000|           554|       500|              0.554|0.9025270758122743|
+-----------+--------------+----------+-------------------+------------------+



# Testing EASE

In [7]:
dataset = utils.InteractionsDataset(interactions=df)
model = models.EASE(
    inn2id=dataset.inn2id,
    id2inn=dataset.id2inn,
)
model.fit(X=dataset.sparse_matrix)

In [72]:
model.predict_for_kt(
    user_inn=5136003767,
    interactions_set=dataset.interaction_sets[5136003767]
)

,inn_kt,inn_dt,score
64,5136003767,6232756495,0.040369
0,5136003767,5136003767,0.039903
14,5136003767,2848219711,0.023562
31,5136003767,1247180228,0.017421
15,5136003767,6906968318,0.013021
...,...,...,...
19,5136003767,7595035291,-0.003252
44,5136003767,3159757135,-0.003292
34,5136003767,9389563811,-0.003482
50,5136003767,9326675863,-0.003662
